# Lab 10 · Auditoría de sesgos y riesgos del modelo telco
### Tema 10 · MU Gestión de Sistemas Complejos · Caso transversal: operador de telecomunicaciones

**Entorno:** SageMaker Notebook (kernel Python 3) · **Librerías:** pandas, matplotlib
**Duración estimada:** 150–180 min · **Nivel:** intermedio-avanzado

---

## 1. Idea general del laboratorio

Cierra la serie con una pregunta obligatoria: **¿es justa, fiable y responsable la inteligencia que hemos
construido?** Un modelo puede tener una *accuracy* excelente y, a la vez, funcionar mucho peor para ciertos
grupos, usar variables **proxy** de atributos sensibles y desencadenar decisiones que perjudican a quienes
ya estaban en desventaja. Trabajaremos con las **predicciones de un modelo de churn** (como el del Lab 4) y
la verdad conocida, más atributos de grupo (región y zona). Mediremos disparidades de rendimiento y de
error, detectaremos proxies y razonaremos sobre **gobernanza**.

### Conceptos clave
- **Paridad demográfica:** tasas de predicción positiva similares entre grupos.
- **Igualdad de oportunidades:** **recall** similar entre grupos.
- (Son métricas a veces **incompatibles**; elegir es una decisión ética, no solo técnica.)

> **Nota para el vídeo.** El lab original usa **Athena (SQL)**; aquí los equivalentes en **pandas** (con el
> SQL en comentarios). Dataset `telco_model_audit_lab10.csv` (1000 filas) con un **sesgo inyectado**: el
> modelo acierta mucho más en unas regiones que en otras. Hay que detectarlo y cuantificarlo.

In [ ]:
# ===========================================================================
# CELDA 1 · Dependencias
# ===========================================================================
import sys
!{sys.executable} -m pip install -q matplotlib pandas
print("Dependencias listas")

In [ ]:
# ===========================================================================
# CELDA 2 · Cargar el dataset de auditoria
# ===========================================================================
import pandas as pd, numpy as np, matplotlib.pyplot as plt

RUTA = '.'   # local; o 's3://<tu-bucket>/audit'
df = pd.read_csv(f'{RUTA}/telco_model_audit_lab10.csv')
# Columnas: region, zone_type, tenure_months, monthly_price + y_true (verdad),
# y_pred (prediccion 0/1) y y_score (probabilidad estimada por el modelo).
print("Filas:", df.shape[0])
df.head()

## Parte A · Rendimiento global y desagregado

### A.1 · Accuracy global (engañosa)
```sql
SELECT AVG(CASE WHEN y_true=y_pred THEN 1.0 ELSE 0 END) FROM audit;
```

In [ ]:
# ===========================================================================
# CELDA 3 · Accuracy global
# ===========================================================================
# 'acierto' = 1 cuando la prediccion coincide con la verdad, 0 si no.
df['acierto'] = (df['y_true'] == df['y_pred']).astype(int)
# La media de aciertos = accuracy global (parece buena... pero esconde la disparidad).
print("Accuracy global:", round(df['acierto'].mean(), 3))

### A.2 · Accuracy por región (reveladora)
```sql
SELECT region, COUNT(*), AVG(CASE WHEN y_true=y_pred THEN 1.0 ELSE 0 END)
FROM audit GROUP BY region ORDER BY 3 ASC;
```

In [ ]:
# ===========================================================================
# CELDA 4 · Accuracy por region (aqui aparece el sesgo)
# ===========================================================================
acc_region = df.groupby('region').agg(
    clientes=('customer_id', 'count'),   # tamano del grupo
    accuracy=('acierto', 'mean'),        # accuracy dentro del grupo
).round(3).sort_values('accuracy')       # de peor a mejor
print(acc_region)

> **Atención.** La accuracy global **esconde** la disparidad. Solo al desagregar se ve que el modelo va
> muy bien en unas regiones y mal en otras. Un modelo "bueno en promedio" puede ser **inaceptable** para la
> región peor servida.

### A.3 · Por tipo de zona

In [ ]:
# ===========================================================================
# CELDA 5 · Accuracy por tipo de zona (urbana/periurbana/rural)
# ===========================================================================
acc_zona = df.groupby('zone_type').agg(
    clientes=('customer_id', 'count'),
    accuracy=('acierto', 'mean'),
).round(3).sort_values('accuracy')
print(acc_zona)

## Parte B · Disparidad en los errores

Importa **cómo** se equivoca: un falso positivo (predecir abandono que no ocurre) y un falso negativo (no
detectar un abandono real) tienen consecuencias distintas y su reparto puede ser injusto.

In [ ]:
# ===========================================================================
# CELDA 6 · Falsos positivos/negativos, recall y precision por region
# ===========================================================================
def metricas_grupo(g):
    # Matriz de confusion del grupo:
    tp = ((g.y_pred == 1) & (g.y_true == 1)).sum()   # verdaderos positivos
    fp = ((g.y_pred == 1) & (g.y_true == 0)).sum()   # falsos positivos
    fn = ((g.y_pred == 0) & (g.y_true == 1)).sum()   # falsos negativos
    # recall = de los abandonos REALES, cuantos detecto el modelo.
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    # precision = de lo que marco como abandono, cuanto acerto.
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    return pd.Series({'falsos_positivos': fp, 'falsos_negativos': fn,
                      'recall': round(recall, 3), 'precision': round(precision, 3)})

# apply por grupo (include_groups=False evita incluir la columna de agrupacion).
err = df.groupby('region').apply(metricas_grupo, include_groups=False).sort_values('recall')
print(err)

**Preguntas.** ¿Dónde se escapan más abandonos reales (**recall** bajo)? Si la campaña de retención sigue
al modelo, ¿qué clientes quedan **sistemáticamente desatendidos**?

## Parte C · Métricas formales de equidad

### C.1 · Paridad demográfica
```sql
SELECT region, AVG(CAST(y_pred AS double)) FROM audit GROUP BY region;
```

In [ ]:
# ===========================================================================
# CELDA 7 · Paridad demografica: tasa de positivos por region
# ===========================================================================
# Media de y_pred por region = proporcion de clientes marcados "en riesgo".
# Si difiere mucho entre regiones, el modelo "selecciona" desigualmente.
tasa_pos = df.groupby('region')['y_pred'].mean().round(3).sort_values(ascending=False)
print("Tasa de positivos por region (paridad demografica):")
print(tasa_pos)
print("\nBrecha de tasa de positivos:", round(tasa_pos.max() - tasa_pos.min(), 3))

### C.2 · Igualdad de oportunidades (brecha de recall)

In [ ]:
# ===========================================================================
# CELDA 8 · Igualdad de oportunidades: recall por region y su brecha
# ===========================================================================
def recall_grupo(g):
    pos = g[g['y_true'] == 1]                          # solo abandonos reales
    return (pos['y_pred'] == 1).mean() if len(pos) else float('nan')

rec = df.groupby('region').apply(recall_grupo, include_groups=False)
print(rec.round(3).sort_values())
# La diferencia entre el mejor y el peor recall mide directamente la inequidad:
# cuanto mayor, mas desigual es la PROTECCION que ofrece el modelo por region.
print("\nBrecha de recall (igualdad de oportunidades):", round(rec.max() - rec.min(), 3))

In [ ]:
# ===========================================================================
# CELDA 9 · Visual: accuracy y recall por region, lado a lado
# ===========================================================================
fig, ax = plt.subplots(1, 2, figsize=(13, 5))   # dos graficos en una fila
# Barras horizontales de accuracy por region (ordenadas).
acc_region['accuracy'].sort_values().plot.barh(ax=ax[0], color='#2980b9')
ax[0].set_title('Accuracy por region'); ax[0].set_xlim(0, 1)
# Barras horizontales de recall por region.
rec.sort_values().plot.barh(ax=ax[1], color='#c0392b')
ax[1].set_title('Recall por region (igualdad de oportunidades)'); ax[1].set_xlim(0, 1)
plt.tight_layout(); plt.show()

> **Atención.** Una brecha de recall grande significa que el modelo **protege mucho mejor** a unas zonas
> que a otras. Si coinciden con renta o ruralidad, está reproduciendo una **desigualdad territorial**.

## Parte D · Variables proxy

`region` y `zone_type` pueden codificar implícitamente el nivel socioeconómico: actuarían como **proxy** de
un atributo sensible aunque el modelo no lo use directamente.

In [ ]:
# ===========================================================================
# CELDA 10 · ¿Predice mas abandono en zonas rurales? (proxy)
# ===========================================================================
proxy = df.groupby('zone_type').agg(
    churn_real=('y_true', 'mean'),       # abandono real por zona
    churn_predicho=('y_pred', 'mean'),   # abandono que predice el modelo
    precio_medio=('monthly_price', 'mean'),
    accuracy=('acierto', 'mean'),
).round(3).sort_values('churn_predicho', ascending=False)
print(proxy)

**El bucle de retroalimentación del sesgo.** Si el modelo predice peor en una región, la retención la
atiende peor, esos clientes se van más, el histórico futuro registra aún más abandono allí, y el siguiente
modelo aprende que esa región es "irrecuperable" y la desatiende todavía más. Reconocer este bucle es el
corazón del laboratorio.

In [ ]:
# ===========================================================================
# CELDA 11 · Resumen de auditoria: ¿coinciden peor accuracy, recall y ruralidad?
# ===========================================================================
resumen = acc_region.copy()
resumen['recall'] = rec.round(3)                 # recall por region
resumen['tasa_positivos'] = tasa_pos.round(3)    # tasa de positivos por region
# % de clientes en zona rural por region (proxy socioeconomico).
pct_rural = df.assign(rural=(df.zone_type == 'rural').astype(int)).groupby('region')['rural'].mean()
resumen['pct_rural'] = pct_rural.round(3)
print(resumen.sort_values('accuracy'))   # ordenado de peor a mejor accuracy

## Parte E · (Opcional) SageMaker Clarify y gobernanza

**SageMaker Clarify** automatiza métricas de sesgo y explica predicciones (SHAP). **No sustituye el
juicio**: señala dónde mirar. **Gobernanza mínima:** *model card* (datos, métricas por grupo,
limitaciones), supervisión humana, monitorización de deriva, mitigación (reentrenar equilibrado, recalibrar
umbrales por grupo, limitar uso) y responsabilidad asignada.

In [ ]:
# ===========================================================================
# CELDA 12 · Borrador automatico de "model card" con los hallazgos
# ===========================================================================
# idxmin/idxmax devuelven la region con peor/mejor accuracy.
peor = resumen['accuracy'].idxmin(); mejor = resumen['accuracy'].idxmax()
print("=== FICHA DE AUDITORIA (borrador de model card) ===")
print(f"Accuracy global:           {df['acierto'].mean():.3f}")
print(f"Mejor region:              {mejor} (acc {resumen.loc[mejor,'accuracy']:.3f}, "
      f"recall {resumen.loc[mejor,'recall']:.3f})")
print(f"Peor region:               {peor} (acc {resumen.loc[peor,'accuracy']:.3f}, "
      f"recall {resumen.loc[peor,'recall']:.3f})")
print(f"Brecha de recall:          {rec.max()-rec.min():.3f}")
print(f"Brecha de accuracy:        {resumen['accuracy'].max()-resumen['accuracy'].min():.3f}")
print(f"Accuracy en zona rural:    {df[df.zone_type=='rural']['acierto'].mean():.3f}")
print(f"Accuracy en zona urbana:   {df[df.zone_type=='urbana']['acierto'].mean():.3f}")
print("Recomendacion: NO desplegar sin mitigacion en las regiones de peor recall.")

## Parte F · Entregable · Parte G · Cierre

Entregable (5–7 págs.): accuracy global vs desagregada (región y zona); disparidad de errores (FP/FN,
recall, precision); paridad demográfica e igualdad de oportunidades con la **brecha de recall**; una
**variable proxy** y su riesgo; el **bucle de retroalimentación**; mitigación (≥2 medidas) y gobernanza; y
un **veredicto razonado**: ¿desplegarías el modelo tal cual, con condiciones, o no? Al terminar, limpia
recursos (si usaste Athena/Clarify) y **Stop**.

---

### Cierre didáctico
No solo "¿funciona?", sino **"¿funciona para todos, y qué pasa si se equivoca?"**. Una accuracy global
brillante puede ocultar que el modelo **abandona a las regiones peor servidas**; mediste la equidad,
desenmascaraste proxies y comprendiste el **bucle que amplifica la desigualdad**. La lección final: **la
técnica y la responsabilidad son inseparables.**